### CPE371: Big Data Engineering
### Lab 7.2: House Prices with Spark DataFrames (Lab 2)

Run the Lab 2 tutorial on the two branches' house datasets, then answer the two Lab 2 questions with DataFrame code:

| # | question | output |
| - | -------- | ------ |
| 1 | Summarized pricing table across `MSZoning` | `MSZoning, n, avg_price, max_price, min_price, avg_lot` |
| 2 | The 5 highest house prices per square foot | `Id, MSZoning, YearBuilt, SalePrice, TotalArea, PricePerSqft` |

The same logic is packaged as `src/house_analytics.py` (run `python src/house_analytics.py`) and tested in
`tests/test_house_analytics.py`. The last cell checks that the notebook and the module give identical tables.

### Part 0: Spark setup
The lab VM kernel predefines `spark`; locally we create it. The data is the 4 CSVs in `lab+files+and+datasets/`.

In [1]:
import os
import sys
import tempfile

ROOT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SRC_DIR = os.path.join(ROOT_DIR, "src")
DATA_DIR = os.path.join(ROOT_DIR, "lab+files+and+datasets")
sys.path.insert(0, SRC_DIR)
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # workers use this kernel's Python
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")      # local mode: skip the hostname-lookup warning

from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("CPE371_Lab7_2_HousePrices")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false")
         # toPandas() below: Arrow needs pyarrow, which the course environment does not install
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
path = lambda name: os.path.join(DATA_DIR, name)
print(f"Spark {spark.version} | data {os.path.relpath(DATA_DIR, ROOT_DIR)}: {sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.csv'))}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/28 19:39:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0 | data lab+files+and+datasets: ['Branch1_GovDataset.csv', 'Branch1_HouseDataset.csv', 'Branch2_GovDataset.csv', 'Branch2_HouseDataset.csv']


### Part 1: Lab 2 tutorial (DataFrames)
The original tutorial cells, run against the local copies of the datasets.

Two cells needed a change for Spark 4, which enables ANSI mode by default (the lab VM runs an older Spark):
1. `cast("integer")` on the string `'65.0'` raises `CAST_INVALID_INPUT` instead of returning 65. We cast to `double` first, then to `integer`.
2. `+` on string columns raises `DATATYPE_MISMATCH` instead of implicitly casting them. We cast the three area columns to `int` before adding them.

In [2]:
house_1_df = spark.read.format("csv").option("header", "true").option("delimiter", ",").load(path("Branch1_HouseDataset.csv"))
gov_1_df = spark.read.format("csv").option("header", "true").option("delimiter", ",").load(path("Branch1_GovDataset.csv"))
house_2_df = spark.read.format("csv").option("header", "true").option("delimiter", ",").load(path("Branch2_HouseDataset.csv"))
gov_2_df = spark.read.format("csv").option("header", "true").option("delimiter", ",").load(path("Branch2_GovDataset.csv"))
print(house_1_df.columns)
print(gov_1_df.columns, gov_1_df.count())

['Id', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF']
['Id', 'LotFrontage', 'MSZoning', 'YearBuilt', 'SaleCondition', 'SalePrice', 'Label'] 1000


In [3]:
gov_1_df.printSchema()          # every column is a string until we cast it
gov_1_df.collect()[0]['LotFrontage']

root
 |-- Id: string (nullable = true)
 |-- LotFrontage: string (nullable = true)
 |-- MSZoning: string (nullable = true)
 |-- YearBuilt: string (nullable = true)
 |-- SaleCondition: string (nullable = true)
 |-- SalePrice: string (nullable = true)
 |-- Label: string (nullable = true)



'65.0'

In [4]:
gov_1_df.sort(gov_1_df['LotFrontage']).show(5)
gov_1_df.sort(gov_1_df['LotFrontage'].desc()).show(5)

+---+-----------+--------+---------+-------------+---------+-----+
| Id|LotFrontage|MSZoning|YearBuilt|SaleCondition|SalePrice|Label|
+---+-----------+--------+---------+-------------+---------+-----+
| 25|       NULL|      RL|     1968|       Normal|   154000|    Y|
| 15|       NULL|      RL|     1960|       Normal|   157000|    Y|
| 13|       NULL|      RL|     1962|       Normal|   144000|    Y|
|  8|       NULL|      RL|     1973|       Normal|   200000|    Y|
| 32|       NULL|      RL|     1966|       Normal|   149350|    Y|
+---+-----------+--------+---------+-------------+---------+-----+
only showing top 5 rows
+---+-----------+--------+---------+-------------+---------+-----+
| Id|LotFrontage|MSZoning|YearBuilt|SaleCondition|SalePrice|Label|
+---+-----------+--------+---------+-------------+---------+-----+
|322|       99.0|      RL|     2004|       Normal|   354000|    Y|
|206|       99.0|      RL|     1990|       Normal|   180500|    Y|
| 28|       98.0|      RL|     2007|  

In [5]:
print(gov_1_df.select(['Id', 'LotFrontage']).columns)
print(gov_1_df.drop('SalePrice').columns)

['Id', 'LotFrontage']
['Id', 'LotFrontage', 'MSZoning', 'YearBuilt', 'SaleCondition', 'Label']


In [6]:
df_1 = gov_1_df.join(house_1_df, 'Id', 'inner')
df_2 = gov_2_df.join(house_2_df, 'Id', 'inner')
print("df_1 " + str(df_1.count()) + " rows")
print("df_2 " + str(df_2.count()) + " rows")
df = df_1.unionAll(df_2)
print("df " + str(df.count()) + " rows")

df_1 1000 rows


df_2 460 rows


df 1460 rows


In [7]:
print("LotFrontage > 60:", df.filter(df['LotFrontage'] > 60.0).count())
print("not null:", df.filter(df['LotFrontage'].isNotNull()).count(),
      "| null:", df.filter(df['LotFrontage'].isNull()).count(),
      "| dropna():", df.dropna().count())

LotFrontage > 60: 754


not null: 1201 | null: 259 | dropna(): 1201


In [8]:
from pyspark.sql.functions import avg, mean, count, stddev, sum, col
df.agg(stddev('LotFrontage')).show()

+-------------------+
|stddev(LotFrontage)|
+-------------------+
|  24.28475177448321|
+-------------------+



In [9]:
_df = df.dropna().drop('Id', 'SaleCondition', 'SalePrice', 'Label', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF')
_df = _df.withColumn("LotFrontage", col("LotFrontage").cast("double").cast("integer"))   # Spark 4: via double
_df = _df.withColumn("YearBuilt", col("YearBuilt").cast("integer"))
_df.groupby('MSZoning').mean().show()

+--------+-----------------+------------------+
|MSZoning| avg(LotFrontage)|    avg(YearBuilt)|
+--------+-----------------+------------------+
|      RL|74.68004338394793| 1975.711496746204|
|      FV|59.49122807017544|2004.8947368421052|
|      RH|58.92307692307692|            1948.0|
|      RM|52.36683417085427| 1940.788944723618|
| C (all)|             69.7|            1924.1|
+--------+-----------------+------------------+



In [10]:
df2 = df.dropna().drop('Id', 'SaleCondition', 'SalePrice', 'Label', 'MSZoning', 'LotFrontage', 'YearBuilt')
# Spark 4: cast before adding; the lab VM's Spark added the strings implicitly and printed 2566.0
df2 = df2.withColumn("TotalArea", df["TotalBsmtSF"].cast("int") + df['1stFlrSF'].cast("int") + df['2ndFlrSF'].cast("int"))
df2.show(5)

+-----------+--------+--------+---------+
|TotalBsmtSF|1stFlrSF|2ndFlrSF|TotalArea|
+-----------+--------+--------+---------+
|        856|     856|     854|     2566|
|       1262|    1262|       0|     2524|
|        920|     920|     866|     2706|
|        756|     961|     756|     2473|
|       1145|    1145|    1053|     3343|
+-----------+--------+--------+---------+
only showing top 5 rows


In [11]:
df.toPandas().head()

,Id,LotFrontage,MSZoning,YearBuilt,SaleCondition,SalePrice,Label,TotalBsmtSF,1stFlrSF,2ndFlrSF
0,1,65.0,RL,2003,Normal,208500,Y,856,856,854
1,2,80.0,RL,1976,Normal,181500,Y,1262,1262,0
2,3,68.0,RL,2001,Normal,223500,Y,920,920,866
3,4,60.0,RL,1915,Abnorml,140000,Y,756,961,756
4,5,84.0,RL,2000,Normal,250000,Y,1145,1145,1053


In [12]:
# The tutorial saves into the working directory; a temp dir keeps the repo clean.
out_dir = os.path.join(tempfile.mkdtemp(), "preprocessed_data")
df.write.format("csv").options(delimiter=',').options(header=True).save(out_dir)
print("written", len([f for f in os.listdir(out_dir) if f.endswith('.csv')]), "part file(s) to", out_dir)

written 2 part file(s) to /var/folders/8c/w9ly1qwx6r940zyhccx3k3g80000gn/T/tmp8zj45xdh/preprocessed_data


---
### Preparing the data for both questions

1. **Join, then union by name.** Each branch splits a house across two files that share `Id`, so an inner join on `Id` rebuilds the full record. The tutorial stacks the branches with `unionAll`, which matches columns by **position**. `unionByName` matches them by **name**, so a branch file with its columns in a different order cannot silently put `LotFrontage` values into `MSZoning`.
2. **Cast.** `spark.read.csv` without a schema reads every column as `string`. On a string column, `max` and `min` compare text, not numbers, so `'99900'` beats `'755000'` (next cell). Spark 4 also refuses arithmetic on strings. So the numeric columns are cast to `int` or `double` first.

In [13]:
from pyspark.sql.functions import max as spark_max, min as spark_min, round as spark_round

houses = df_1.unionByName(df_2)
for name in ["Id", "YearBuilt", "SalePrice", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]:
    houses = houses.withColumn(name, col(name).cast("int"))
houses = houses.withColumn("LotFrontage", col("LotFrontage").cast("double")).cache()

print(houses.count(), "houses")
print(dict(houses.dtypes))
print("max SalePrice as string:", df.agg(spark_max("SalePrice")).first()[0],
      "| as int:", houses.agg(spark_max("SalePrice")).first()[0])

1460 houses
{'Id': 'int', 'LotFrontage': 'double', 'MSZoning': 'string', 'YearBuilt': 'int', 'SaleCondition': 'string', 'SalePrice': 'int', 'Label': 'string', 'TotalBsmtSF': 'int', '1stFlrSF': 'int', '2ndFlrSF': 'int'}


max SalePrice as string: 99900 | as int: 755000


---
### Question 1: Summarized pricing table across `MSZoning`

**Approach.** Use `groupBy("MSZoning")` with one `agg` that computes all five columns in a single pass.

- **Rule 1: keep every house, no `dropna()`.** 259 houses have no `LotFrontage`. The tutorial's `dropna()` would remove them from *every* column, including `n` and the prices. The expected `n` values add up to 1460, which is all houses. `avg("LotFrontage")` already skips NULLs, so `avg_lot` is still correct without dropping anything.
- **Rule 3: two decimals with `decimal(10,2)`.** `round(x, 2)` returns a double, and Spark prints `74528.0`. The expected table shows `74528.00`. Casting to `decimal(10,2)` rounds half-up and always prints two decimals.
- **Sort order:** by `avg_price` descending, which gives the expected FV, RL, RH, RM, C (all).

In [14]:
q1 = (houses.groupBy("MSZoning")
          .agg(count("*").alias("n"),
               avg("SalePrice").cast("decimal(10,2)").alias("avg_price"),
               spark_max("SalePrice").alias("max_price"),
               spark_min("SalePrice").alias("min_price"),
               avg("LotFrontage").cast("decimal(10,2)").alias("avg_lot"))
          .orderBy(col("avg_price").desc(), col("MSZoning")))
q1.show()

+--------+----+---------+---------+---------+-------+
|MSZoning|   n|avg_price|max_price|min_price|avg_lot|
+--------+----+---------+---------+---------+-------+
|      FV|  65|214014.06|   370878|   144152|  59.49|
|      RL|1151|191004.99|   755000|    39300|  74.68|
|      RH|  16|131558.38|   200000|    76000|  58.92|
|      RM| 218|126316.83|   475000|    37900|  52.37|
| C (all)|  10| 74528.00|   133900|    34900|  69.70|
+--------+----+---------+---------+---------+-------+



In [15]:
# Why rule 1 and rule 3 matter: the same summary with dropna() and round()
(houses.dropna().groupBy("MSZoning")
       .agg(count("*").alias("n"), spark_round(avg("SalePrice"), 2).alias("avg_price"))
       .orderBy(col("avg_price").desc())
       .show())

+--------+---+---------+
|MSZoning|  n|avg_price|
+--------+---+---------+
|      FV| 57| 217721.3|
|      RL|922|192281.88|
|      RH| 13|130610.31|
|      RM|199|125467.98|
| C (all)| 10|  74528.0|
+--------+---+---------+



**Answer.** The table matches the expected output exactly. FV (Floating Village) has the highest average price at 214,014.06 over 65 houses. RL (low-density residential) is by far the largest zone, with 1151 houses and the most expensive single house (755,000). C (all) (commercial) is the cheapest, averaging 74,528.00.

The second table shows what the two rules prevent. With `dropna()`, every zone loses the houses without a `LotFrontage`, so `n` drops from 1460 to 1201 in total and the averages shift. With `round()`, C (all) prints as `74528.0`.

---
### Question 2: The 5 highest house prices per square foot

**Approach.**
1. **Rule 6.** `TotalArea = TotalBsmtSF + 1stFlrSF + 2ndFlrSF`. The columns are already `int`, so the sum is an `int` (`827`). The lab VM's Spark added the raw strings as doubles and printed `827.0`.
2. **Rule 7.** `PricePerSqft = round(SalePrice / TotalArea, 2)`, cast to `decimal(10,2)` as in Q1. Houses with `TotalArea = 0` are dropped, so the division is always defined.
3. **Rule 5: only `SaleCondition == "Normal"`.** A *Partial* sale is a new home sold before it was finished, so its price is set against a floor area that is not complete. Those sales dominate an unfiltered ranking, as the next cell shows. Filtering on Normal sales compares like with like and gives the expected table.
4. Sort by `PricePerSqft` descending, with `Id` as a tie-breaker, and keep the top 5.

In [16]:
ppsf = (houses.withColumn("TotalArea", col("TotalBsmtSF") + col("1stFlrSF") + col("2ndFlrSF"))
              .filter(col("TotalArea") > 0)
              .withColumn("PricePerSqft", spark_round(col("SalePrice") / col("TotalArea"), 2).cast("decimal(10,2)")))
columns = ["Id", "MSZoning", "YearBuilt", "SalePrice", "TotalArea", "PricePerSqft"]

print("Without the SaleCondition filter:")
ppsf.orderBy(col("PricePerSqft").desc(), col("Id")).select(*columns, "SaleCondition").show(5)

Without the SaleCondition filter:
+---+--------+---------+---------+---------+------------+-------------+
| Id|MSZoning|YearBuilt|SalePrice|TotalArea|PricePerSqft|SaleCondition|
+---+--------+---------+---------+---------+------------+-------------+
|689|      RL|     2007|   392000|     2838|      138.13|      Partial|
|899|      RL|     2009|   611657|     4694|      130.31|      Partial|
|533|      RL|     1955|   107500|      827|      129.99|       Normal|
|804|      RL|     2008|   582933|     4556|      127.95|      Partial|
|393|      RL|     1959|   106500|      882|      120.75|       Normal|
+---+--------+---------+---------+---------+------------+-------------+
only showing top 5 rows


In [17]:
q2 = (ppsf.filter(col("SaleCondition") == "Normal")
          .select(*columns)
          .orderBy(col("PricePerSqft").desc(), col("Id"))
          .limit(5))
q2.show()

+---+--------+---------+---------+---------+------------+
| Id|MSZoning|YearBuilt|SalePrice|TotalArea|PricePerSqft|
+---+--------+---------+---------+---------+------------+
|533|      RL|     1955|   107500|      827|      129.99|
|393|      RL|     1959|   106500|      882|      120.75|
|534|      RL|     1946|    39300|      334|      117.66|
|186|      RM|     1892|   475000|     4143|      114.65|
|873|      RL|     1953|   116000|     1015|      114.29|
+---+--------+---------+---------+---------+------------+



**Answer.** The table matches the expected output exactly. The top Normal sale is house 533 (RL, built 1955), at 129.99 per square foot: 107,500 for 827 sq ft. Four of the five are small, older RL houses, where a modest price over a small area gives a high rate. House 186 is the exception. It is a large RM house from 1892 (4143 sq ft) whose 475,000 price keeps it at 114.65.

Without the filter, Partial sales 689, 899 and 804 (built 2007–2009) enter the top 5, because their prices are set against unfinished floor areas.

---
### Cross-check with `src/house_analytics.py`
The packaged module must produce the same two tables as the step-by-step code above.

In [18]:
import house_analytics as ha

module_df = ha.cast_columns(ha.load_house_data(spark, DATA_DIR))
assert module_df.count() == houses.count() == 1460
assert ha.mszoning_summary(module_df).collect() == q1.collect()
assert ha.top_price_per_sqft(module_df).collect() == q2.collect()
print("module and notebook agree on Q1 and Q2")

module and notebook agree on Q1 and Q2


In [19]:
spark.stop()